# Notebook 3 · Society Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Toy Use Case

---

## What is a Society Architecture?

In a **society MAS** agents interact according to explicit *social rules* rather
than authority relationships.
Each agent (a *citizen*) follows its own rule and casts a public vote.
The group converges when a majority (or consensus) emerges from the voting history.

```
         ┌──────────────────────────────────────┐
         │          PUBLIC VOTING BOARD          │
         │  (all votes visible to all citizens)  │
         └────┬──────────┬──────────┬────────────┘
              │          │          │
       ┌──────▼──┐  ┌────▼──┐  ┌───▼──────┐  ┌──────────┐
       │ Budget  │  │Comfort│  │ Culture  │  │Simplicity│
       │Citizen  │  │Citizen│  │ Citizen  │  │ Citizen  │
       └─────────┘  └───────┘  └──────────┘  └──────────┘
            each citizen follows one social rule → public vote
```

### Key Properties
| Property | Society |
|---|---|
| Authority | None — social rules govern behaviour |
| Communication | Citizen → public board |
| Stop condition | Majority vote (≥ 3 of 4 citizens agree) |
| Failure mode | Perpetual split vote if rules are too divergent |

### When to Use It
Society patterns suit tasks where you want **diverse evaluation criteria**
to be weighed openly — for example, recommending one option from a set
of pre-built alternatives when no single agent should decide alone.

---

## What We Will Build

We pre-build three candidate travel packages (one per destination).
Four **citizens**, each following a different social rule, inspect the packages
and vote publicly.
The package with the most votes after each round wins if it reaches majority,
otherwise voting continues.

| Citizen | Social rule |
|---|---|
| Budget citizen | Prefer the lowest reasonable total cost |
| Comfort citizen | Prefer direct flights and minimal friction |
| Culture citizen | Prefer the richest food and cultural activity mix |
| Simplicity citizen | Prefer the most coherent 4-day plan |


## 1 · Setup: One Model, One Request, One Catalog

The same three blocks appear in every notebook of this series:

| Block | Purpose |
|---|---|
| `model` | One small, cheap LLM used by every agent |
| `USER_REQUEST` | The single traveler request we want to fulfill |
| `CATALOG` + helpers | The only data source agents are allowed to use |

Keeping these identical lets you compare orchestration patterns in isolation.
Nothing changes between notebooks except *how agents talk to each other*.

> **Prerequisite:** set `OPENAI_API_KEY` in your environment before running.


In [1]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# ── One model powers every agent in the series ──────────────────────────────
# We use gpt-4.1-mini for speed and low cost during learning.
# Swap to any model that supports structured output (temperature=0 keeps
# outputs deterministic so you can rerun and compare results).
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# ── The traveler request ─────────────────────────────────────────────────────
# All four notebooks solve this exact request.
# The only thing that changes is the orchestration architecture.
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use destinations, flights, hotels, and activities
# listed below. This constraint keeps the toy example clean and comparable.

DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",         "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",         "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",          "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",          "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",      "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel",   "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
# Simple filter functions so agents can look up options by destination.

def flights_for(destination: str) -> list[dict]:
    return [f for f in FLIGHTS if f["destination"] == destination]

def hotels_for(destination: str) -> list[dict]:
    return [h for h in HOTELS if h["destination"] == destination]

def activities_for(destination: str) -> list[dict]:
    return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    """Return the full catalog as a plain-text string suitable for a prompt."""
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Build Candidate Packages and Define Citizens

### Why pre-build packages?

The society pattern shines when agents evaluate *existing options* rather than
building a plan from scratch.
Pre-building three concrete packages (one per destination) gives the citizens
something fixed to debate — mimicking a real recommendation scenario.

### Why use a Vote schema?

A `Vote` schema with a `confidence` field lets us inspect not just *what*
citizens chose but *how strongly* they believed in it.
This richer signal could be used to weight votes, though in this tutorial
we use simple majority for clarity.


In [2]:
from typing import Literal
from collections import Counter

# ── Build one candidate package per destination ───────────────────────────────
# Each package combines the cheapest flight, the most central hotel,
# and two representative activities for that destination.
# This is deterministic — the society's job is to evaluate, not to design.

def build_package(destination: str) -> dict:
    # Sort by price to pick the cheapest flight option.
    cheapest_flight = sorted(flights_for(destination), key=lambda f: f["price"])[0]
    # Sort by price to pick the most affordable hotel.
    cheapest_hotel  = sorted(hotels_for(destination), key=lambda h: h["price_per_night"])[0]
    # Take the first two activities (one food, one culture per destination in our catalog).
    selected_acts   = activities_for(destination)[:2]
    return {
        "destination": destination,
        "period":      DESTINATIONS[destination]["best_period"],
        "flight":      cheapest_flight,
        "hotel":       cheapest_hotel,
        "activities":  selected_acts,
    }

PACKAGES = {
    "package_1": build_package("Lisbon"),
    "package_2": build_package("Barcelona"),
    "package_3": build_package("Prague"),
}

def packages_text() -> str:
    """Render all packages as a plain-text summary for citizen prompts."""
    lines = []
    for pkg_id, pkg in PACKAGES.items():
        lines.append(f"{pkg_id}: {pkg['destination']}")
        lines.append(f"  Period  : {pkg['period']}")
        lines.append(
            f"  Flight  : {pkg['flight']['code']} | {pkg['flight']['type']} "
            f"| EUR {pkg['flight']['price']}"
        )
        lines.append(
            f"  Hotel   : {pkg['hotel']['name']} | EUR {pkg['hotel']['price_per_night']}/night"
        )
        lines.append(
            "  Activities: " + ", ".join(a["name"] for a in pkg["activities"])
        )
        lines.append("")
    return "\n".join(lines).strip()

PACKAGES_TEXT = packages_text()

# ── Vote schema ────────────────────────────────────────────────────────────────
# package_id   = the citizen's choice (machine-readable Literal)
# reason       = the citizen's justification (human-readable trace)
# confidence   = how strongly the citizen believes in its choice (1–5)

class Vote(BaseModel):
    package_id:  Literal["package_1", "package_2", "package_3"]
    reason:      str = Field(description="One sentence explaining this citizen's choice.")
    confidence:  int = Field(description="Confidence from 1 (low) to 5 (high).", ge=1, le=5)

voter_llm = model.with_structured_output(Vote)

# ── Citizen registry ───────────────────────────────────────────────────────────
# Each citizen follows exactly one social rule.
# The rule is visible on the public board, so other citizens can anticipate
# how this citizen will vote — a key property of a transparent society.
CITIZENS = {
    "Budget citizen":     "Vote for the package with the lowest reasonable total trip cost.",
    "Comfort citizen":    "Vote for the package with the most direct flights and least friction.",
    "Culture citizen":    "Vote for the package with the richest mix of food and cultural activities.",
    "Simplicity citizen": "Vote for the most coherent and easy-to-follow 4-day plan.",
}

# ── Citizen voter function ─────────────────────────────────────────────────────
def cast_vote(citizen_name: str, social_rule: str, public_board: str) -> Vote:
    """
    Ask one citizen to cast a vote.

    Society properties visible here:
    - The citizen sees the public board (all previous votes) before deciding.
    - The citizen follows its own social rule, not a manager's instruction.
    - The citizen cannot modify the packages — only evaluate and vote.
    """
    return voter_llm.invoke([
        SystemMessage(content=(
            "You are a citizen in a society-style travel MAS.\n"
            "No manager controls you. You follow your social rule and vote publicly.\n"
            "Inspect the candidate packages and the current public board, "
            "then vote for exactly one package.\n\n"
            f"Your social rule: {social_rule}"
        )),
        HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Candidate packages:\n{PACKAGES_TEXT}\n\n"
            f"Public board (all previous votes):\n{public_board or 'No votes cast yet.'}\n\n"
            "Cast your vote."
        )),
    ])

print("Packages built:", list(PACKAGES.keys()))
print("Citizens:", list(CITIZENS.keys()))


Packages built: ['package_1', 'package_2', 'package_3']
Citizens: ['Budget citizen', 'Comfort citizen', 'Culture citizen', 'Simplicity citizen']


## 3 · Run the Voting Rounds

This is the core of the society architecture.

### Consensus logic
- Each citizen votes in every round.
- All votes are immediately visible to subsequent citizens in the same round.
- After each round, we check for a **majority** (≥ 3 of 4 citizens agree).
- If majority is reached, the society stops. Otherwise voting continues.

> This transparency — each citizen sees how others voted — is what makes it
> a *society* rather than a simple parallel vote: citizens can shift if they
> see strong consensus forming, but their social rule ultimately governs.


In [3]:
public_board = []    # grows with every vote cast
tally = Counter()    # tracks votes in the current round
winning_package = None
MAX_ROUNDS = 3

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n--- Voting round {round_num} ---")
    tally = Counter()  # reset tally for this round

    for citizen_name, social_rule in CITIZENS.items():
        # ── Cast one vote ──────────────────────────────────────────────────
        # The citizen sees every vote already on the board this round,
        # not just its own private context.
        ballot = cast_vote(citizen_name, social_rule, "\n".join(public_board))

        # ── Record the vote publicly ───────────────────────────────────────
        entry = (
            f"Round {round_num} | {citizen_name} → {ballot.package_id} "
            f"(confidence {ballot.confidence}/5): {ballot.reason}"
        )
        public_board.append(entry)
        tally[ballot.package_id] += 1
        print(f"  {citizen_name}: {ballot.package_id} (conf {ballot.confidence})")

    # ── Majority check ─────────────────────────────────────────────────────
    # The society rule: if any package gets ≥ 3 votes in a single round, stop.
    best_pkg, vote_count = tally.most_common(1)[0]
    print(f"  Tally: {dict(tally)} — leader: {best_pkg} ({vote_count} votes)")

    if vote_count >= 3:
        winning_package = best_pkg
        print(f"  Majority reached! Winner: {winning_package}")
        break

# If no round produced a majority, pick the most-voted package overall.
if winning_package is None:
    winning_package = tally.most_common(1)[0][0]
    print(f"\nNo majority — selecting plurality winner: {winning_package}")

# ── Final summary ──────────────────────────────────────────────────────────────
# Produce a human-readable explanation of the outcome.
final_summary = model.invoke([
    SystemMessage(content=(
        "Summarise the society's decision in plain language. "
        "Explain which package won, why the citizens converged on it, "
        "and what the winning package offers the traveler."
    )),
    HumanMessage(content=(
        f"Traveler request:\n{USER_REQUEST}\n\n"
        f"Public board:\n{'\n'.join(public_board)}\n\n"
        f"Winning package details:\n{PACKAGES[winning_package]}"
    )),
]).content

print("\n" + "="*60)
print("PUBLIC VOTING BOARD")
print("="*60)
for entry in public_board:
    print(" •", entry)

print("\n" + "="*60)
print(f"WINNING PACKAGE: {winning_package}")
print("="*60)
pkg = PACKAGES[winning_package]
print(f"  Destination : {pkg['destination']}")
print(f"  Period      : {pkg['period']}")
print(f"  Flight      : {pkg['flight']['code']} | {pkg['flight']['type']} | EUR {pkg['flight']['price']}")
print(f"  Hotel       : {pkg['hotel']['name']} | EUR {pkg['hotel']['price_per_night']}/night")
print(f"  Activities  : {', '.join(a['name'] for a in pkg['activities'])}")

print("\n" + "="*60)
print("SOCIETY SUMMARY")
print("="*60)
print(final_summary)



--- Voting round 1 ---
  Budget citizen: package_3 (conf 5)
  Comfort citizen: package_3 (conf 5)
  Culture citizen: package_2 (conf 5)
  Simplicity citizen: package_3 (conf 5)
  Tally: {'package_3': 3, 'package_2': 1} — leader: package_3 (3 votes)
  Majority reached! Winner: package_3

PUBLIC VOTING BOARD
 • Round 1 | Budget citizen → package_3 (confidence 5/5): Prague offers the lowest total cost with direct flights and affordable hotel, fitting the mid-range budget and easy travel requirements.
 • Round 1 | Comfort citizen → package_3 (confidence 5/5): Package 3 offers direct flights and the lowest friction, matching the traveler's preference for easy flights and a simple plan.
 • Round 1 | Culture citizen → package_2 (confidence 5/5): Barcelona offers a rich mix of food with tapas evenings and cultural activities like Sagrada Família and modernism routes, fitting the traveler's desire for food and culture.
 • Round 1 | Simplicity citizen → package_3 (confidence 5/5): Prague offers

## 4 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| Agents follow **social rules**, not instructions | Authority is replaced by transparent norms |
| All votes are **publicly visible** | Later citizens can see the emerging consensus |
| The group stops by **majority**, not a manager | Decentralised stopping condition |
| Candidates are **pre-built** | The society evaluates options, it doesn't design them |

### How this differs from the other notebooks
- **vs Flat (nb 1):** agents evaluate fixed options instead of building a plan collaboratively
- **vs Hierarchical (nb 2):** no manager decides; the group converges through open voting
- **vs Team (nb 4):** there are no complementary roles; all citizens do the same *type* of job

### When the society pattern struggles
- **Split votes** can persist if citizen rules are orthogonal (e.g., cheapest vs highest quality)
- **Cascade effects** — early votes can unduly influence later citizens even when the board
  shows only a narrow lead
- **Pre-built packages** limit the solution space; creative combinations are impossible
